# Exercise: keeping .Stat Suite in sync with FMR

Publish real ECB exchange-rate structures to an FMR and from the FMR to
.Stat Suite, then remove from .Stat the structures that the FMR no
longer has. The course notebook, `fmr_stat_sync.ipynb`, explains the
steps.

The exercise takes about 20 minutes. Run the cells in order. Five steps
are marked *to complete*: their cell is empty, and the notes and the
hint say what to write. The other cells are already written. The answer
is in `fmr_stat_sync_practice.py`, in the folder of this notebook.

## Setup

The configuration and the helpers are in `helpers.py`, in the folder of
this notebook. The settings are read from environment variables when it
is imported, so they must be set before the imports. Each participant
has an SDMX Lab instance and its credentials: the second cell sets
`PREFIX` and requests them, unless they are already set in the
environment; the passwords are not displayed. The other settings can be set in the environment or in the
same cell (e.g. `os.environ["NSI"] = "..."`). The kernel must be
restarted to change a setting after the imports.

| Variable | Meaning | Default |
|---|---|---|
| `PREFIX` | Prefix of the ids of the structures | `PX_SYNC` |
| `SDMX_LAB` | Identifier of the SDMX Lab instance, the first part of its address (`<id>.sdmx.solutions`) | requested |
| `FMR_USER` and `FMR_PASSWORD`, or `FMR_TOKEN` | Credentials of the FMR | requested |
| `STAT_USER` and `STAT_PASSWORD` | Credentials of .Stat (Keycloak) | requested |
| `DOTSTAT_TOKEN_FILE` | File with a .Stat token, used without `STAT_USER` | `dotstat.token` |
| `FMR_ENDPOINT` | Web service of the FMR | derived from `SDMX_LAB` |
| `NSI`, `TRANSFER`, `SPACE` | .Stat web services and dataspace | the SIS-CC demo |
| `KEYCLOAK`, `STAT_CLIENT_ID` | Keycloak realm and client of .Stat | the SIS-CC demo, `app` |
| `AGENCY` | Agency of the structures | `BIS` |

`fmr_client()` and `stat_uploader()` return the clients with write
access, `fmr_service()` and `stat_service()` the readers, and
`build_structures(ecb)` the structures to publish.

In [ ]:
%pip install --quiet ./pysdmx-1.24.0rc1-py3-none-any.whl[all]
%pip install --quiet certifi

In [ ]:
import os
from getpass import getpass

# Every structure of the tutorial starts with this prefix, which tells them
# apart from the other structures of the agency. Only letters, digits and
# underscores are allowed.
os.environ["PREFIX"] = "PX_SYNC"

# The SDMX Lab instance and the credentials of the participant are
# requested, unless they are already set in the environment. The passwords
# are not displayed. An empty answer for .Stat reuses the credentials of
# the FMR.
if not (os.environ.get("SDMX_LAB") or os.environ.get("FMR_ENDPOINT")):
    os.environ["SDMX_LAB"] = input("SDMX Lab instance (<id>.sdmx.solutions): ")
if not (os.environ.get("FMR_USER") or os.environ.get("FMR_TOKEN")):
    os.environ["FMR_USER"] = input("FMR user: ")
    os.environ["FMR_PASSWORD"] = getpass("FMR password: ")
if not os.environ.get("STAT_USER"):
    user, password = input(".Stat user: "), getpass(".Stat password: ")
    os.environ["STAT_USER"] = user or os.environ.get("FMR_USER", "")
    os.environ["STAT_PASSWORD"] = password or os.environ.get(
        "FMR_PASSWORD", ""
    )

In [ ]:
# Some imports stay unused, and some names undefined, until the exercise
# is completed.
# ruff: noqa: F401, F821
from contextlib import suppress
from io import BytesIO

from helpers import (
    AGENCY,
    DELETION_ORDER,
    ECB_STRUCTURES,
    PREFIX,
    STAT_FORMAT,
    build_structures,
    check_connections,
    fmr_client,
    fmr_service,
    list_urns,
    stat_service,
    stat_uploader,
)

from pysdmx.api.fmr.maintenance import StructureAction
from pysdmx.api.qb import (
    StructureDetail,
    StructureQuery,
    StructureReference,
    StructureType,
)
from pysdmx.errors import Invalid, NotFound
from pysdmx.io import read_sdmx
from pysdmx.model import AgencyScheme, Code, Codelist

## Part 1 · Publish to the FMR

### Step 1 · Check the connections.

`check_connections` tests the FMR and .Stat with read-only queries. Both services must be connected, and the agency must be registered in both, before going on.

If a service fails, correct its settings, restart the kernel and run the cells again.

In [ ]:
check_connections()

### Step 2 · Read the ECB exchange-rate structures and prepare the structures to publish.

`build_structures` copies the ECB codelists and concepts to the agency of the participant, adds `PREFIX` to their ids, and builds two dataflows on top of them: EXR (in `keep`) and EXR_OLD (in `legacy`).

`legacy` stands for structures that have been retired in the FMR but remain on .Stat.

In [ ]:
ecb = read_sdmx(ECB_STRUCTURES, validate=False)
keep, legacy = build_structures(ecb)
print("keep:  ", [s.short_urn for s in keep])
print("legacy:", [s.short_urn for s in legacy])

### Step 3 (to complete) · Upload the structures to keep to the FMR.

`Replace` overwrites the existing structures, so the step can be run again. `legacy` is not uploaded.

**Hint:** `fmr_client()` returns a `RegistryMaintenanceClient`. Call its `put_structures` method with `keep` and `action=StructureAction.Replace`.

In [ ]:
# To be completed

### Step 4 (to complete) · Download the EXR dataflow with all its references from the FMR.

The dataflow comes with its DSD, the concept scheme, the codelists and the SDMX:AGENCIES agency scheme.

**Hint:** Build a `StructureQuery` with `artefact_type=StructureType.DATAFLOW`, `agency_id=AGENCY`, `resource_id=f"{PREFIX}_DF_EXR"`, `version="1.0"`, `detail=StructureDetail.FULL` and `references=StructureReference.ALL`. Send it with `fmr_service().structure(query)`, which returns bytes, and read them with `read_sdmx(BytesIO(...), validate=False)`.

**Result:** `graph`.

In [ ]:
# To be completed

## Part 2 · Publish from the FMR to .Stat

### Step 5 · Start from a clean .Stat.

.Stat rejects a structure that already exists, so the structures left by a previous run are removed first. Each list ends with its DSD and its dataflow, so reversing it deletes the dependents first.

In [ ]:
stat = stat_uploader()
for artefact in [*reversed(legacy), *reversed(keep)]:
    with suppress(Invalid, NotFound):
        stat.delete_structure(artefact)

### Step 6 (to complete) · Publish the download to .Stat.

The SDMX:AGENCIES agency scheme is left out, so that it does not overwrite the one on .Stat.

The result has `success`, which must be True, and `messages`: if .Stat rejects a structure, the reason is in the messages.

To run the step again, run Step 5 first: .Stat rejects the structures that already exist.

**Hint:** Drop the `AgencyScheme` from `graph.structures`, and call `stat_uploader().submit_structure` with the other structures and `STAT_FORMAT`. Print the `success` and the `messages` of the result.

In [ ]:
# To be completed

### Step 7 · Create a difference between the FMR and .Stat.

`legacy` is submitted to .Stat only.

In [ ]:
result = stat_uploader().submit_structure(legacy, STAT_FORMAT)
print(result.success, result.messages)

## Part 3 · Bring .Stat back in sync

### Step 8 · List the structures on both sides.

`list_urns` requests the structures of the agency one type at a time, as stubs, and returns the short URNs of those whose id starts with `PREFIX`.

The FMR is the reference: without valid credentials its list is empty, and every structure on .Stat would then be taken as retired, so the step stops in that case.

In [ ]:
fmr_urns = list_urns(fmr_service())
if not fmr_urns:
    raise RuntimeError("The FMR list is empty: check the FMR credentials.")
stat_urns = list_urns(stat_service())
print("FMR:  ", sorted(fmr_urns))
print(".Stat:", sorted(stat_urns))

### Step 9 (to complete) · Find what .Stat has and the FMR does not.

The extras must be inspected before anything is deleted: they are the EXR_OLD dataflow, its DSD and CL_UNIT. The codelists shared with EXR are still in the FMR, so they are not part of the extras.

**Hint:** Subtract `fmr_urns` from `stat_urns` and print the result.

**Result:** `extras`.

In [ ]:
# To be completed

### Step 10 · Try to delete the DSD first.

.Stat refuses (409), because the EXR_OLD dataflow still uses the DSD.

In [ ]:
try:
    stat_uploader().delete_structure(
        [u for u in extras if u.startswith("DataStructure=")]
    )
except Invalid as error:
    print(error)

### Step 11 (to complete) · Delete the extras, dependents first.

`delete_structure` takes a list of short URNs, deletes them in order and stops at the first error. In that case, fix the cause and run it again with the remaining structures.

**Hint:** `DELETION_ORDER` gives the position of each type (constraints, dataflows, DSDs, concept schemes, codelists). Sort `extras` with the key `DELETION_ORDER[urn.split("=")[0]]` and pass the list to `stat_uploader().delete_structure`.

In [ ]:
# To be completed

### Step 12 · Check that .Stat matches the FMR again, and remove the structures to keep from .Stat.

The printed list must be empty: nothing extra is left on .Stat.

The structures to keep are removed from .Stat, so that the exercise can be run again. The copies in the FMR remain, because pysdmx cannot delete from the FMR.

In [ ]:
stat_after = list_urns(stat_service())
print(sorted(stat_after - fmr_urns))
stat_uploader().delete_structure(list(reversed(keep)))

### Bonus · Create another difference and remove it.

A codelist that only exists on .Stat is found by the comparison, like the structures in `legacy`.

**Hint:** Build a `Codelist` with `agency=AGENCY`, an id that starts with `PREFIX` and an underscore, a name and two `Code` items, each with an id and a name. Submit it to .Stat only, and repeat the comparison and the deletion.

In [ ]:
# To be completed